In [1]:
from pathlib import Path

import pandas as pd

from data_cleaning.utils import peek

CLEANED_DIR = Path("cleaned_data")

## Load transcriptomics datasets

In [2]:
t2019 = pd.read_parquet(CLEANED_DIR / "transcriptomics_2019_UGA_cleaned.parquet")
t2020 = pd.read_parquet(CLEANED_DIR / "transcriptomics_2020_UGA_cleaned.parquet")
tsdy2867 = pd.read_parquet(CLEANED_DIR / "transcriptomics_SDY2867_cleaned.parquet")

print("2019_UGA:", t2019.shape)
print("2020_UGA:", t2020.shape)
print("SDY2867: ", tsdy2867.shape)

2019_UGA: (275, 64125)
2020_UGA: (48, 32709)
SDY2867:  (73, 109805)


## Combine into one dataframe

In [3]:
transcriptomics = pd.concat([t2019, t2020, tsdy2867], ignore_index=True)
print("Combined shape:", transcriptomics.shape)
peek(transcriptomics)

Combined shape: (396, 109805)


,participant_id,TRAN_ENSG00000000003_d0,TRAN_ENSG00000000005_d0,TRAN_ENSG00000000419_d0,TRAN_ENSG00000000457_d0,TRAN_ENSG00000000460_d0,TRAN_ENSG00000000938_d0,TRAN_ENSG00000000971_d0,TRAN_ENSG00000001036_d0,TRAN_ENSG00000001084_d0,TRAN_ENSG00000001167_d0,TRAN_ENSG00000001460_d0,TRAN_ENSG00000001461_d0,TRAN_ENSG00000001497_d0,TRAN_ENSG00000001561_d0,TRAN_ENSG00000001617_d0,TRAN_ENSG00000001626_d0,TRAN_ENSG00000001629_d0,TRAN_ENSG00000001630_d0,TRAN_ENSG00000001631_d0
0,2019_UGA.ID_001,1.001284,-0.123083,0.651376,0.546033,1.491505,-0.622255,-1.007577,0.598130,0.109457,0.388386,0.512635,1.018868,0.923870,0.007177,-0.310983,-0.184023,0.478017,0.767752,0.390241
1,2019_UGA.ID_005,0.586238,-0.123083,-0.682261,0.408613,0.482631,-0.929775,-0.492218,0.042291,0.819701,0.915067,0.428503,1.455299,1.194103,0.816484,0.350294,-0.184023,1.078850,0.598956,0.949021
2,2019_UGA.ID_008,0.268739,-0.123083,0.012367,0.200666,-0.070990,0.235228,-0.282735,0.089897,1.102819,-0.055020,1.049170,0.256959,0.117698,-0.487343,-0.310983,-0.184023,-0.535201,0.102128,-0.233563
3,2019_UGA.ID_011,-0.473472,-0.123083,-0.478352,0.620830,1.193616,0.432613,-0.732424,0.266911,1.158720,1.180399,-0.197465,0.385646,-0.057288,1.445018,-0.310983,-0.184023,1.007912,0.360122,0.999589
4,2019_UGA.ID_014,-1.145122,-0.123083,-0.007073,0.811627,0.804596,0.083290,-0.379761,0.929377,0.935906,1.007133,1.064422,1.308638,0.622721,1.428517,2.138805,-0.184023,0.931346,0.754686,0.775816


## Missingness check

In [4]:
d0_cols = [c for c in transcriptomics.columns if c.endswith("_d0")]
d7_cols = [c for c in transcriptomics.columns if c.endswith("_d7")]
n = len(transcriptomics)

d0_present = transcriptomics[d0_cols].notna().any(axis=1).sum()
d7_present = transcriptomics[d7_cols].notna().any(axis=1).sum()

print(f"d0 present rows: {d0_present}/{n}")
print(f"d7 present rows: {d7_present}/{n}")

d0 present rows: 395/396
d7 present rows: 235/396


## PCA

In [5]:
from sklearn.decomposition import PCA

N_COMPONENTS = 50

def cohort_pca(df, suffix, n_components):
    cols = [c for c in df.columns if c.endswith(suffix)]
    complete_cols = [c for c in cols if df[c].notna().all()]
    if not complete_cols:
        return None, None
    X = df[complete_cols].values
    n = min(n_components, X.shape[0] - 1, X.shape[1])
    pca = PCA(n_components=n, random_state=42)
    label = suffix.lstrip("_")
    pc_df = pd.DataFrame(
        pca.fit_transform(X),
        columns=[f"TRAN_PC_{label}_{i+1}" for i in range(n)]
    )
    pc_df.insert(0, "participant_id", df["participant_id"].values)
    print(f"    {len(df)} subjects, {len(complete_cols)} genes → {n} components ({pca.explained_variance_ratio_.sum():.1%} var)")
    return pc_df, pca

print("d0 PCA per cohort:")
d0_pcs = []
for name, df in [("2019_UGA", t2019), ("2020_UGA", t2020), ("SDY2867", tsdy2867)]:
    print(f"  {name}")
    pc_df, _ = cohort_pca(df, "_d0", N_COMPONENTS)
    if pc_df is not None:
        d0_pcs.append(pc_df)

d0_pc_df = pd.concat(d0_pcs, ignore_index=True)
print(f"\nd0 shape: {d0_pc_df.shape}")

d0 PCA per cohort:
  2019_UGA
    275 subjects, 31222 genes → 50 components (71.9% var)
  2020_UGA
    48 subjects, 27936 genes → 47 components (100.0% var)
  SDY2867

d0 shape: (323, 51)


In [6]:
print("d7 PCA per cohort:")
d7_pcs = []
for name, df in [("2019_UGA", t2019), ("2020_UGA", t2020), ("SDY2867", tsdy2867)]:
    d7_cols_cohort = [c for c in df.columns if c.endswith("_d7")]
    has_d7 = df[d7_cols_cohort].notna().any(axis=1)
    if not has_d7.any():
        print(f"  {name}: no d7 data, skipping")
        continue
    print(f"  {name}")
    pc_df, _ = cohort_pca(df[has_d7].reset_index(drop=True), "_d7", N_COMPONENTS)
    if pc_df is not None:
        d7_pcs.append(pc_df)

d7_pc_df = pd.concat(d7_pcs, ignore_index=True) if d7_pcs else None
if d7_pc_df is not None:
    print(f"\nd7 shape: {d7_pc_df.shape}")

d7 PCA per cohort:
  2019_UGA
    163 subjects, 28028 genes → 50 components (70.1% var)
  2020_UGA: no d7 data, skipping
  SDY2867
    72 subjects, 37514 genes → 50 components (79.0% var)

d7 shape: (235, 51)


In [7]:
all_ids = transcriptomics[["participant_id"]]
tran_pca = all_ids.merge(d0_pc_df, on="participant_id", how="left")
if d7_pc_df is not None:
    tran_pca = tran_pca.merge(d7_pc_df, on="participant_id", how="left")
print(tran_pca.shape)
peek(tran_pca)

(396, 101)


,participant_id,TRAN_PC_d0_1,TRAN_PC_d0_2,TRAN_PC_d0_3,TRAN_PC_d0_4,TRAN_PC_d0_5,TRAN_PC_d0_6,TRAN_PC_d0_7,TRAN_PC_d0_8,TRAN_PC_d0_9,TRAN_PC_d0_10,TRAN_PC_d0_11,TRAN_PC_d0_12,TRAN_PC_d0_13,TRAN_PC_d0_14,TRAN_PC_d0_15,TRAN_PC_d0_16,TRAN_PC_d0_17,TRAN_PC_d0_18,TRAN_PC_d0_19
0,2019_UGA.ID_001,54.857104,-2.327803,20.658080,1.290844,22.602886,35.865655,-0.217187,12.459896,10.779050,9.981861,-3.306802,-7.826256,0.051140,17.817827,6.751547,-1.165548,3.410936,10.274941,1.023394
1,2019_UGA.ID_005,64.340100,-5.530560,-24.225036,-11.234761,-5.973387,27.053021,16.420792,9.297817,-1.582333,3.867934,4.422988,12.087987,-2.581871,-11.557819,-0.547310,8.387108,-7.969929,7.834185,-6.899411
2,2019_UGA.ID_008,14.242545,-12.435488,9.568582,2.040647,12.556634,-1.800915,-9.591414,-2.575313,3.695120,4.900290,-8.797805,-9.064617,0.193539,12.285878,-17.141893,-2.175617,-0.789818,10.811512,-0.079176
3,2019_UGA.ID_011,46.799769,-11.102347,-34.414588,-8.874203,-24.431189,-18.968893,1.865773,3.228261,-0.344186,-3.115868,-3.224383,9.580202,-3.638812,6.257673,0.310771,-4.026654,3.929481,1.140762,-0.758605
4,2019_UGA.ID_014,66.204774,-4.364196,-40.787463,-8.972190,-11.719986,-8.043705,8.899763,5.921097,-1.965794,-0.797008,-2.287930,13.054126,-2.837833,-6.953112,-5.744330,3.673071,-5.072188,5.253703,-3.839770


## Save

In [8]:
tran_pca.to_parquet(CLEANED_DIR / "transcriptomics_pca.parquet", index=False)
print("Saved to cleaned_data/transcriptomics_pca.parquet")

Saved to cleaned_data/transcriptomics_pca.parquet
